# Лабораторная работа #5. Задача кластеризации

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pydantic.v1.datetime_parse import time_expr
import time
from sklearn.cluster import AgglomerativeClustering, AffinityPropagation
from sklearn.datasets import make_classification, make_blobs
import seaborn as sns
from sklearn.metrics import silhouette_score, adjusted_rand_score, adjusted_mutual_info_score, v_measure_score
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score
from sklearn.mixture import GaussianMixture
from sklearnex.cluster import KMeans
from sklearnex.neighbors import NearestNeighbors

RANDOM_STATE = 13

## Генерация датасетов

### make_classification


#### 1. Датасет с 3 кластерами


In [ ]:
data_x1, data_y1 = make_classification(n_samples=300, n_classes=3, n_features=2,
                                       n_clusters_per_class=1, random_state=151,
                                       n_redundant=0, class_sep=1.5)

sns.scatterplot(x=data_x1[:, 0], y=data_x1[:, 1],
                palette="Set2", hue=data_y1)

plt.title("Dataset 1: 3 clusters")



#### 2. Датасет с 4 кластерами


In [ ]:
data_x2, data_y2 = make_classification(n_samples=200, n_features=2, n_classes=4,
                                       n_clusters_per_class=1, n_informative=2,
                                       n_redundant=0, random_state=1, class_sep=2)

sns.scatterplot(x=data_x2[:, 0], y=data_x2[:, 1], hue=data_y2, palette="Set2")

plt.title("Dataset 2: 4 clusters")


#### 3. Датасет с 5 кластерами


In [ ]:
import plotly.express as px
data_x3, data_y3 = make_classification(n_samples=250, n_features=3, n_redundant=0,
                                       n_classes=5, n_clusters_per_class=1,
                                       n_informative=3,
                                       random_state=15, class_sep=2)

fig = px.scatter_3d(x=data_x3[:, 0], y=data_x3[:, 1], z=data_x3[:, 2],
                    color=data_y3, title="Dataset 3: 5 clusters", width=500, height=400)

fig.update_traces(marker=dict(size=5, line=dict(width=1, color='Black')))


### make_blobs

In [ ]:
data_x4, data_y4 = make_blobs(n_samples=100, n_features=2, centers=3, random_state=135)
sns.scatterplot(x=data_x4[:, 0], y=data_x4[:, 1], hue=data_y4, palette="bright")

In [ ]:
data_x5, data_y5 = make_blobs(n_samples=350, n_features=2, centers=5, random_state=167)
sns.scatterplot(x=data_x5[:, 0], y=data_x5[:, 1], hue=data_y5, palette="bright")

In [ ]:
data = pd.read_csv("datasets/neo.csv", index_col="Unnamed: 0")
data_X, data_y = data.drop("hazardous", axis=1), data["hazardous"]
data_X

## Решение задачи кластеризации

In [ ]:
metrics = ["ARI", "V-Measure", "AMI", # внешние
           "Silhouette Coefficient", "Davies-Bouldin Index", "Calinski-Harabasz Index" ] # внутренние

results_table = pd.DataFrame(columns=["algorithm", "dataset_title"]+metrics)

def get_table_metrics(X, y, labels, dataset_number, algorithm):
    ari = round(adjusted_rand_score(y, labels), 2)
    ami = round(adjusted_mutual_info_score(y, labels), 2)
    v_score = round(v_measure_score(y, labels), 2)

    s_score = round(silhouette_score(X, labels), 2)
    davies_bouldin_index = round(davies_bouldin_score(X, labels), 2)
    calinski_harabasz_index = round(calinski_harabasz_score(X, labels), 2)

    results_table.loc[len(results_table)] = [algorithm, dataset_number,
                                             ari, ami, v_score,
                                             s_score, davies_bouldin_index, calinski_harabasz_index]

In [ ]:
datasets = [(data_x1, data_y1), (data_x2, data_y2), (data_x3, data_y3),
            (data_x4, data_y4), (data_x5, data_y5), (data_X.values, data_y.values)]
true_number_of_clusters = [3, 4, 5, 3, 5, 2]
datasets_names = [f"Dataset {i + 1}" for i in range(len(datasets) - 1)] + ["Neo dataset"]

def make_graph():
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    axes = axes.flatten()

    for ax, number_of_cls in zip(axes, true_number_of_clusters):
        ax.set_title(f"Generated dataset with {number_of_cls} clusters in true")

    axes[-1].set_title("Real data with 2 clusters in true")

    return fig, axes

### k-means

In [ ]:
fig, axes = make_graph()

sns.set_palette("mako")

for (x, y), ax in zip(datasets, axes):
    inertias = []
    for i in range(2, 13):
        model = KMeans(n_clusters=i, n_jobs=-1, n_init="auto", random_state=13)
        model.fit(x)
        inertias.append(model.inertia_)
    sns.lineplot(x=range(2, 13), y=inertias, ax=ax, marker="o")

fig.suptitle("Elbow Method for k-means clustering")


На основе метрики компактности (метод локтя) были получены следующие оптимальные параметры кластеров для датасетов (оптимальные точки определяются через "локтевые" точки, то в точках с резкими изгибами ломаной.

* Датасет 1 - 3
* Датасет 2 - 4
* Датасет 3 - 5
* Датасет 4 - 4
* Датасет 5 - 5
* Датасет с реальными данными - 5

In [ ]:
fig, axes = make_graph()

sns.set_palette("mako")

for (x, y), ax in zip(datasets, axes):
    silhouette_scores = []
    for i in range(2, 13):
        model = KMeans(n_clusters=i, n_jobs=-1, n_init="auto", random_state=13)
        model.fit(x)

        score = silhouette_score(x, model.fit_predict(x))
        silhouette_scores.append(score)

    sns.lineplot(x=range(2, 13), y=silhouette_scores, ax=ax, marker="o")

fig.suptitle("Silhouette Method for k-means clustering")

С помощью метода силуэта были получены следующие оптимумы:

* Датасет 1 - 3
* Датасет 2 - 4
* Датасет 3 - 5
* Датасет 4 - 2
* Датасет 5 - 5
* Датасет с реальными данными - 2


Количество кластеров отличается для датасета номер 4 и датасета с реальными данными.

В силу того, что метод силуэта более точен, будем использовать параметры, полученные с помощью него.

| Датасет         | Метод локтя | Метод силуэта |
|:----------------|:-----------:|--------------:|
| Датасет 1       |      3      |             3 |
| Датасет 2       |      4      |             4 |
| Датасет 3       |      5      |             5 |
| Датасет 4       |      4      |             2 |
| Датасет 5       |      5      |             5 |
| Реальные данные |      5      |             2 |


In [ ]:
silhouette_opt = [3, 4, 5, 2, 5, 2] # результаты метода силуэта
fig, axes = make_graph()
kmeans_models = []


for (X, y), name, k, ax in zip(datasets, datasets_names, silhouette_opt, axes):
    model = KMeans(n_clusters=k, random_state=15, n_jobs=-1)
    start = time.time()
    model.fit(X)
    finish = time.time()
    kmeans_models.append(model)

    labels = model.predict(X)
    get_table_metrics(X, y, labels, name, "kmeans")

    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=labels, s=30, ax=ax, palette="Set2")

    centers = model.cluster_centers_

    ax.scatter(centers[:, 0], centers[:, 1], c="red")

    print(f"Time taken: {finish - start} for {name}")

plt.suptitle("K-means clustering")

In [ ]:
results_table

In [ ]:
import plotly.graph_objects as go

labels = kmeans_models[2].predict(data_x3)
centers = kmeans_models[2].cluster_centers_
fig=px.scatter_3d(x=data_x3[:, 0], y = data_x3[:, 1], z=data_x3[:, 2], color = labels, width=500, height=400, title="Kmeans clustering")
fig.add_trace(
    go.Scatter3d(
        x=centers[:, 0], 
        y=centers[:, 1], 
        z=centers[:, 2],
        mode='markers',
        marker=dict(
            size=15,
            color='red',
            line=dict(width=2, color='white')
        ),
        name='Centroids'
    )
)
fig.update_layout(coloraxis_showscale=False)


In [ ]:
px.scatter_matrix(
    data_X,
    dimensions=data_X.columns, 
    color=data_y,
    title="Реальное разделение данных",
    color_discrete_sequence=px.colors.qualitative.Vivid,
    width=900,
    height=900
)


In [ ]:
centers = kmeans_models[-1].cluster_centers_
labels = kmeans_models[-1].predict(data_X)
df_centers = pd.DataFrame(centers, columns=data_X.columns)


df_total = pd.concat([data_X, df_centers])
all_labels = list(labels.astype(str)) + ['Center'] * len(centers)


px.scatter_matrix(
    df_total,
    dimensions=data_X.columns,
    color=all_labels,
    symbol=all_labels,
    title="Кластеры и центры",
    width=900,
    height=900
)


In [ ]:
data_X_with_kmeans_labels = data_X.copy()
data_X_with_kmeans_labels["labels"] = labels
data_X_with_kmeans_labels.groupby("labels").agg(["count", "mean", "std", "min", "max"]).T

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 10))

for ax, col in zip(axes, data_X.columns):
    sns.barplot(x="labels", y=col, data=data_X_with_kmeans_labels, ax=ax, palette="Set2")

### DBSCAN

In [ ]:
import numpy as np

fig, axes = make_graph()

for (x, y), ax in zip(datasets, axes):
    k = 2 * x.shape[1]

    neigh = NearestNeighbors(n_neighbors=k)

    nbrs = neigh.fit(x)
    distances, indices = nbrs.kneighbors(x)

    k_distances = np.sort(distances[:, k-1])

    ax.plot(range(len(k_distances)), k_distances)

fig.suptitle("K-Distance")

In [ ]:
from sklearnex.cluster import DBSCAN

eps_values = [0.4, 0.8, 1.4, 1, 0.75, 0.15]
min_samples_values = [5, 5, 10, 5, 5, 100]
fig, axes = make_graph()
dbscan_models = []


for (X, y), name, eps, ms, ax in zip(datasets, datasets_names, eps_values, min_samples_values, axes):
    model = DBSCAN(eps=eps, min_samples=ms)

    labels = model.fit_predict(X)
    dbscan_models.append(model)
    get_table_metrics(X, y, labels, name, "DBSCAN")

    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=labels, s=30, ax=ax, palette="Set2")

plt.suptitle("DBSCAN clustering")


In [ ]:
labels = dbscan_models[2].labels_
px.scatter_3d(x=data_x3[:, 0], y = data_x3[:, 1], z=data_x3[:, 2],
              color = labels, width=500, height=400,
              title="DBSCAN clustering")

In [ ]:

labels = dbscan_models[-1].labels_

px.scatter_matrix(
    data_X,
    dimensions=data_X.columns,
    color=labels.astype(str),
    title="Кластеры",
    width=900,
    height=900
)

In [ ]:
results_table[results_table["algorithm"] == "DBSCAN"]

In [ ]:
data_X_with_dbscan_labels = data_X.copy()
data_X_with_dbscan_labels["labels"] = labels
data_X_with_dbscan_labels.groupby("labels").agg(["count", "mean", "std", "min", "max"]).T

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 10))

for ax, col in zip(axes, data_X.columns):
    sns.barplot(x="labels", y=col, data=data_X_with_dbscan_labels, ax=ax, palette="Set2")

### Иерархическая кластеризация

In [ ]:
fig, axes = make_graph()

h_models = []

for (X,y), name, n_cl, ax in zip(datasets[:-1], datasets_names[:-1], silhouette_opt[:-1], axes[:-1]):
    model = AgglomerativeClustering(n_clusters=n_cl)

    labels = model.fit_predict(X)

    h_models.append(model)
    get_table_metrics(X, y, labels, name, "Agglomerative Clustering")

    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=labels, s=30, ax=ax, palette="viridis")
axes[-1].remove()
plt.suptitle("Agglomerative Clustering")

In [ ]:

from scipy.cluster.hierarchy import linkage, dendrogram
fig, axes = make_graph()
for (X,y), ax in zip(datasets[:-1], axes[:-1]):
    Z = linkage(X, method='ward')

    dendrogram(Z, truncate_mode='lastp', p=12, ax=ax) # truncate_mode='lastp' схлопывает дерево для читаемости

plt.suptitle("Dendrograms")
axes[-1].remove()

In [ ]:
labels = h_models[2].labels_
px.scatter_3d(x=data_x3[:, 0], y = data_x3[:, 1], z=data_x3[:, 2], color = labels, width=500, height=400, title="Agglomerative Clustering")

In [ ]:
results_table[results_table["algorithm"] == "Agglomerative Clustering"]

### EM-алгоритм

In [ ]:
fig, axes = make_graph()

gm_models = []

for (X,y), name, n_cl, ax in zip(datasets, datasets_names, silhouette_opt, axes):
    model = GaussianMixture(n_components=n_cl, random_state=13)

    labels = model.fit_predict(X)

    gm_models.append(model)
    get_table_metrics(X, y, labels, name, "EM")

    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=labels, s=30, ax=ax, palette="Set2")
plt.suptitle("GaussianMixtures")

In [ ]:
labels = gm_models[2].predict(data_x3)
px.scatter_3d(x=data_x3[:, 0], y = data_x3[:, 1], z=data_x3[:, 2], color = labels, width=500, height=400, title="GaussianMixture")

In [ ]:
labels = gm_models[-1].predict(data_X)

px.scatter_matrix(
    data_X,
    dimensions=data_X.columns,
    color=labels.astype(str),
    title="GaussianMixture",
    width=900,
    height=900
)

In [ ]:
results_table[results_table["algorithm"] == "EM"]

In [ ]:
data_X_with_gm_labels = data_X.copy()
data_X_with_gm_labels["labels"] = labels
data_X_with_gm_labels.groupby("labels").agg(["count", "mean", "std", "min", "max"]).T

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 10))

for ax, col in zip(axes, data_X.columns):
    sns.violinplot(x="labels", y=col, data=data_X_with_gm_labels, ax=ax, palette="Set2")
plt.suptitle("GaussianMixture")

### Affinity Propagation

In [ ]:
fig, axes = make_graph()

af_models = []

for (X,y), name, ax in zip(datasets[:-1], datasets_names[:-1], axes[:-1]):
    model = AffinityPropagation(random_state=13)

    labels = model.fit_predict(X)

    af_models.append(model)
    get_table_metrics(X, y, labels, name, "Affinity Propagation")

    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=labels, s=30, ax=ax, palette="Set2")

plt.suptitle("Affinity Propagation without hyperparameters")
axes[-1].remove()

In [ ]:
fig, axes = make_graph()
preference_values = [-60, -110, -120, -50, -70]
afh_models = []
for (X,y), name, ax, param in zip(datasets[:-1], datasets_names[:-1], axes[:-1], preference_values):
    model = AffinityPropagation(preference=param, random_state=13)

    labels = model.fit_predict(X)

    afh_models.append(model)
    get_table_metrics(X, y, labels, name, "Affinity Propagation")

    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=labels, s=30, ax=ax, palette="Set2")

plt.suptitle("Affinity Propagation with hyperparameters")
axes[-1].remove()

In [ ]:
labels = afh_models[2].predict(data_x3)
px.scatter_3d(x=data_x3[:, 0], y = data_x3[:, 1], z=data_x3[:, 2], color = labels, width=500, height=400,title= "Affinity Propagation")

In [ ]:
results_table[results_table["algorithm"] == "Affinity Propagation"]

## Кастомный kmeans

In [ ]:
custom_table = pd.DataFrame(columns=["algorithm", "dataset_title"]+metrics)
def get_custom_table_metrics(X, y, labels, dataset_number, algorithm):
    ari = round(adjusted_rand_score(y, labels), 2)
    ami = round(adjusted_mutual_info_score(y, labels), 2)
    v_score = round(v_measure_score(y, labels), 2)

    s_score = round(silhouette_score(X, labels), 2)
    davies_bouldin_index = round(davies_bouldin_score(X, labels), 2)
    calinski_harabasz_index = round(calinski_harabasz_score(X, labels), 2)

    custom_table.loc[len(custom_table)] = [algorithm, dataset_number,
                                             ari, ami, v_score,
                                             s_score, davies_bouldin_index, calinski_harabasz_index]

In [ ]:
from custom_algorithms import kmeans
import time
fig, axes = make_graph()
kmeans_custom_models = []

for (X, y), name, k, ax in zip(datasets, datasets_names, silhouette_opt, axes):
    start = time.time()
    model = kmeans(k=k, random_state=15)

    model.fit(X)
    finish = time.time()

    kmeans_custom_models.append(model)

    labels = model.predict(X)
    get_custom_table_metrics(X, y, labels, name, "Custom kmeans")

    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=labels, s=30, ax=ax, palette="Set2")

    centers = model.centroids

    ax.scatter(centers[:, 0], centers[:, 1], c="red")

    print(f"Time taken: {finish - start} for {name}")

plt.suptitle("Custom K-means clustering")

In [ ]:
labels = kmeans_custom_models[2].predict(data_x3)
centers = kmeans_custom_models[2].centroids
fig=px.scatter_3d(x=data_x3[:, 0], y = data_x3[:, 1], z=data_x3[:, 2], color = labels, width=500, height=400, title="Custom kmeans clustering")
fig.add_trace(
    go.Scatter3d(
        x=centers[:, 0],
        y=centers[:, 1],
        z=centers[:, 2],
        mode='markers',
        marker=dict(
            size=15,
            color='red',
            line=dict(width=2, color='white')
        ),
        name='Centroids'
    )
)
fig.update_layout(coloraxis_showscale=False)

In [ ]:
custom_table

In [ ]:
for (X, y), name, model in zip(datasets, datasets_names, kmeans_models):
    labels = model.predict(X)
    get_custom_table_metrics(X, y, labels, name, "sklearn kmeans")

custom_table

In [ ]:
results_table